# 模块一：SwellEx-96 纯 CPU 原始数据预处理与 Google Drive 持久化

本 Notebook 专为 **Google Colab 纯 CPU 运行时**（`Runtime -> Change runtime type -> CPU`）设计，完全不调用 PyTorch / CUDA，避免消耗宝贵的 Colab GPU 计算额度。

### 工作流职责与目录约定
1. **读取原始数据**：从 Google Drive（默认 `/content/drive/MyDrive/SwellEx_Colab/Origindata`）关联 `Origindata/`（约 368 MB，仅需上传一次）。
2. **本地高速盘计算**：先在实例本地 SSD（`<PROJECT_ROOT>/outputs/`）完成 `scipy` / `numpy` 频域特征提取与 `h5py` 写入，规避 Google Drive FUSE 随机写入慢的问题。
3. **落盘至 Google Drive 并生成索引清单**：将生成的各实验包 HDF5 数据集同步到 `/content/drive/MyDrive/SwellEx_Colab/datasets/`，并写入 `dataset_manifest.json` 供 **模块二（`02_network_training_gpu.ipynb`）** 直接读取。

> **注意**：在 Colab 内请直接运行下方单元格调用 Python 数据生成模块，**不要**调用 `./scripts/run.sh`（该脚本用于本地通过 SSH 派发到 `haiqin` 服务器）。

In [1]:
# Cell 1: 环境检测、CPU 运行时提醒、Google Drive 挂载与 Origindata 关联
from pathlib import Path
import importlib.util
import os
import shutil
import subprocess
import sys

IN_COLAB = (
    "google.colab" in sys.modules
    or (importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None)
)

# 1. 检查是否误开了 GPU 运行时，防止白白消耗 Colab GPU 额度
has_nvidia = shutil.which("nvidia-smi") is not None and subprocess.run(
    ["nvidia-smi", "-L"], capture_output=True, text=True
).returncode == 0
if has_nvidia:
    print("⚠️ [额度提醒] 检测到当前实例挂载了 GPU！模块一仅使用纯 CPU（NumPy/SciPy/h5py），")
    print("   建议切换至 Runtime -> Change runtime type -> CPU 以节省 Colab GPU 计算单元。\n")
else:
    print("✅ 当前为纯 CPU 运行环境（零 GPU 额度消耗）。")

# 2. 定位项目根目录（Colab 下若不在仓库内可先 git clone 或解压仓库到 /content/SwellEx）
def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path("/content/SwellEx")]
    for candidate in candidates:
        if (candidate / "scripts" / "remote_workflow" / "packs.py").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "未找到项目根目录（缺少 scripts/remote_workflow/packs.py）。在 Colab 中请先将仓库放在 /content/SwellEx 或切换到仓库目录。"
    )

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 3. 挂载 Google Drive（Colab 环境自动挂载；本地环境回退到项目本地路径以便调试）
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    DRIVE_BASE = Path("/content/drive/MyDrive/SwellEx_Colab")
else:
    DRIVE_BASE = Path(os.environ.get("SWELLEX_DRIVE_SIM", PROJECT_ROOT / "outputs" / "drive_staging"))

DRIVE_ORIGINDATA = DRIVE_BASE / "Origindata"
DRIVE_DATASET_ROOT = DRIVE_BASE / "datasets"
LOCAL_OUTPUTS = PROJECT_ROOT / "outputs"

# 4. 在 Colab 中自动将 Google Drive 里的 Origindata 软链接到 <PROJECT_ROOT>/Origindata
local_origindata = PROJECT_ROOT / "Origindata"
if not local_origindata.exists() and DRIVE_ORIGINDATA.exists():
    local_origindata.symlink_to(DRIVE_ORIGINDATA, target_is_directory=True)
    print(f"🔗 已关联原始数据目录: {local_origindata} -> {DRIVE_ORIGINDATA}")

# 5. 检查数值依赖（Colab 默认自带 numpy / scipy / h5py）
REQUIRED_PKGS = ("numpy", "scipy", "h5py")
MISSING_PKGS = [pkg for pkg in REQUIRED_PKGS if importlib.util.find_spec(pkg) is None]

print(f"PROJECT_ROOT       : {PROJECT_ROOT}")
print(f"Origindata 就绪    : {local_origindata.exists()} ({local_origindata})")
print(f"DRIVE_DATASET_ROOT : {DRIVE_DATASET_ROOT}")
if MISSING_PKGS:
    print(f"ℹ️ 当前环境缺少数值包 {MISSING_PKGS}（后续单元将以配置预览/干跑模式执行；在 Colab 可运行 `!pip install -q h5py numpy scipy`）。")
else:
    print("✅ 数值依赖 (numpy, scipy, h5py) 已全部就绪。")

✅ 当前为纯 CPU 运行环境（零 GPU 额度消耗）。
PROJECT_ROOT       : /Volumes/exDateDisk/projects/SwellEx
Origindata 就绪    : True (/Volumes/exDateDisk/projects/SwellEx/Origindata)
DRIVE_DATASET_ROOT : /Volumes/exDateDisk/projects/SwellEx/outputs/drive_staging/datasets
ℹ️ 当前环境缺少数值包 ['numpy', 'scipy', 'h5py']（后续单元将以配置预览/干跑模式执行；在 Colab 可运行 `!pip install -q h5py numpy scipy`）。

## 1. 选择要生成的实验预设与特征包

下方参数单元直接复用仓库内置的官方 TOML 配置（无硬编码魔法参数）：
- `PRESET`：
  - `"train_before"`：主实验预设（`range_overlap`，最接近点前训练、后测试，共 448 训练窗 / 485 测试窗）
  - `"train_after"`：反向泛化预设（`range_overlap`，最接近点后训练、前测试）
  - `"full"`：整段交错划分预设（`full_interleaved`，4:1 训练/测试）
- `QUICK_SMOKE_END_S`：设为 `None` 生成完整 75 分钟 S5 数据；若首次联调想在几十秒内验证端到端流程，可设为 `299`（仅截取前 300 秒并自动切换到 `full_interleaved`）。
- `MOE_LEARN_MODES`：支持生成 `"demo"`（120 秒快速演示）和 `"before_then_after"`（一份 HDF5 同时支撑 `before_then_after` 与 `distance_holdout` 近/中/远三组持续学习实验）。

In [2]:
# Cell 2: 生成任务参数配置（按需修改）
from copy import deepcopy
import tomllib

# --- 用户可调配置 ---
PRESET = "train_before"            # 可选: "train_before" | "train_after" | "full"
QUICK_SMOKE_END_S = None           # 完整生成保持 None；快速端到端冒烟可设为 299

BUILD_BASE_ROUTES = True           # 生成 RBD / ELM / SCM 单路 HDF5（供 base_pack, data_scale_pack, 单路 MoE 使用）
JOINT_FUSIONS = ["block_rms"]      # 可选子集: ["block_rms", "raw_concat"]（供 joint_feature_pack 全部 7 种组合使用）
BUILD_MOE_RBD_SCM = True           # 生成 MoE_data_scale 专用的 RBD_SCM 联合 HDF5
MOE_LEARN_MODES = [                # 供 MoE_learn 持续学习使用的逐窗去均值 RBD_SCM HDF5:
    "demo",                        #   - "demo": 120s 双阶段快速演示
    "before_then_after",           #   - "before_then_after": 完整 S5，通用于 before_then_after 与 distance_holdout(near/middle/far)
]
OVERWRITE_LOCAL_JOINTS = True      # 若本地临时联合目录已存在，是否先清理重建
# --------------------

PRESET_TOML_MAP = {
    "train_before": PROJECT_ROOT / "scripts/remote_workflow/base_pack/haiqin_range_overlap_train_before.toml",
    "train_after": PROJECT_ROOT / "scripts/remote_workflow/base_pack/haiqin_range_overlap_train_after.toml",
    "full": PROJECT_ROOT / "scripts/remote_workflow/base_pack/haiqin_full_interleaved.toml",
}
MOE_LEARN_TOML_MAP = {
    "demo": PROJECT_ROOT / "scripts/remote_workflow/MoE_learn/haiqin_demo.toml",
    "before_then_after": PROJECT_ROOT / "scripts/remote_workflow/MoE_learn/haiqin_holdout_middle.toml",
}

if PRESET not in PRESET_TOML_MAP:
    raise ValueError(f"未知 PRESET={PRESET!r}，可选: {list(PRESET_TOML_MAP)}")

raw_preset_cfg = tomllib.loads(PRESET_TOML_MAP[PRESET].read_text(encoding="utf-8"))
dataset_cfg = deepcopy(raw_preset_cfg["dataset"])
rbd_cfg = deepcopy(raw_preset_cfg["rbd"])
dataset_tag = PRESET

if QUICK_SMOKE_END_S is not None:
    dataset_cfg["segment_end_s"] = int(QUICK_SMOKE_END_S)
    dataset_cfg["split_strategy"] = "full_interleaved"
    dataset_cfg["manual_dataset_variant_tag"] = f"colab_smoke_{int(QUICK_SMOKE_END_S)}s"
    dataset_tag = f"{PRESET}_smoke_{int(QUICK_SMOKE_END_S)}s"

GENERATION_PAYLOAD = {
    "schema_version": 1,
    "dataset": dataset_cfg,
    "rbd": rbd_cfg,
    "result_path": f"outputs/generation_report_{dataset_tag}.json",
}

print(f"已加载预设配置: {PRESET_TOML_MAP[PRESET].relative_to(PROJECT_ROOT)}")
print(f"  dataset_tag    : {dataset_tag}")
print(f"  split_strategy : {dataset_cfg['split_strategy']} (train_side={dataset_cfg['train_side']})")
print(f"  时间范围       : {dataset_cfg['segment_start_s']}s .. {dataset_cfg['segment_end_s']}s (步长 {dataset_cfg['segment_step_s']}s)")
print(f"  联合特征融合   : {JOINT_FUSIONS}")
print(f"  MoE_data_scale : RBD_SCM={BUILD_MOE_RBD_SCM}")
print(f"  MoE_learn 模式 : {MOE_LEARN_MODES}")

已加载预设配置: scripts/remote_workflow/base_pack/haiqin_range_overlap_train_before.toml
  dataset_tag    : train_before
  split_strategy : range_overlap (train_side=before)
  时间范围       : 0s .. 4499s (步长 1.0s)
  联合特征融合   : ['block_rms']
  MoE_data_scale : RBD_SCM=True
  MoE_learn 模式 : ['demo', 'before_then_after']

## 2. 生成标准单路特征（`RBD` / `ELM` / `SCM`）并执行契约校验

所有写入均在实例本地 `<PROJECT_ROOT>/outputs/Datasets/` 完成，并自动调用 `inspect_h5`、`split_check` 与 `crossroute_check` 验证样本数、频点轴、阵元对一维列主序索引及三路跨方法对齐。

In [3]:
# Cell 3: 提取单路 RBD / ELM / SCM 特征并做 HDF5 结构契约校验
GENERATED_LOCAL: dict[str, dict] = {}

if MISSING_PKGS or not (PROJECT_ROOT / "Origindata").exists():
    print("⏭️ 跳过实际特征计算（缺少数值运行库或 Origindata 未挂载）。配置结构校验通过。")
elif BUILD_BASE_ROUTES or JOINT_FUSIONS or BUILD_MOE_RBD_SCM:
    from scripts.data_processing.base_pack.generation_config import normalize_config
    from scripts.data_processing.base_pack.generation_core import generate_datasets
    from scripts.data_processing.common.dataset_validation import (
        crossroute_check,
        inspect_h5,
        split_check,
    )
    from scripts.data_processing.common.reporting import normalize_expected

    norm_cfg = normalize_config(GENERATION_PAYLOAD, root=PROJECT_ROOT)
    print(f"🚀 开始生成单路 RBD / ELM / SCM HDF5 ({dataset_tag}) ...")
    gen_result = generate_datasets(norm_cfg, root=PROJECT_ROOT)
    route_dirs = {method: Path(path) for method, path in gen_result["datasets"].items()}

    # 对本次生成的三路 HDF5 做单文件、切分不重叠与跨路由一致性契约校验
    expected_cfg = normalize_expected(norm_cfg)
    all_records = []
    for method, folder in route_dirs.items():
        method_records = []
        for h5_path in sorted(p for p in folder.glob("*.h5") if not p.name.startswith("._")):
            rec, _ = inspect_h5(h5_path, method, expected_cfg)
            method_records.append(rec)
            all_records.append(rec)
        split_check(method_records, expected_cfg)
    crossroute_check(all_records)

    rbd_meta = gen_result["dataset_results"]["RBD"]
    GENERATED_LOCAL["base_routes"] = {
        "dataset_tag": dataset_tag,
        "routes": {k: str(v) for k, v in route_dirs.items()},
        "train_count": rbd_meta["train_count"],
        "test_count": rbd_meta["test_count"],
        "frequency_hz": rbd_meta["frequency_hz"],
    }
    print("✅ 单路 HDF5 生成与三路契约验证通过:")
    for method, folder in route_dirs.items():
        print(f"   - {method:4s}: {folder.relative_to(PROJECT_ROOT)}")
    print(
        f"   样本统计: train={rbd_meta['train_count']}, test={rbd_meta['test_count']}, "
        f"频点数={len(rbd_meta['frequency_hz'])}"
    )

⏭️ 跳过实际特征计算（缺少数值运行库或 Origindata 未挂载）。配置结构校验通过。

## 3. 生成联合特征包（`joint_feature_pack`、`MoE_data_scale` 与 `MoE_learn`）

- **`joint_feature_pack`**：基于刚生成的单路 HDF5 拼接全部 7 种组合（`RBD`, `ELM`, `SCM`, `RBD_ELM`, `RBD_SCM`, `ELM_SCM`, `RBD_ELM_SCM`），支持 `block_rms`（仅在训练子集拟合各块均方根缩放）与 `raw_concat`。
- **`MoE_data_scale`**：构建 `RBD_SCM` 双路拼接 HDF5。
- **`MoE_learn`**：使用其独立的逐窗去均值特征提取器（`moe-learn-python-features-v3-window-centered`）生成 `RBD_SCM_train.h5` 与 `RBD_SCM_test.h5`，使模块二无需 `Origindata/` 即可运行全部持续学习阶段。

In [4]:
# Cell 4: 构建 joint_feature_pack / MoE_data_scale / MoE_learn 的联合 HDF5 数据集
MOE_LEARN_STRIP_KEYS = {
    "stage_mode",
    "val_fraction",
    "after_test_fraction",
    "after_grouping",
    "after_range_edges_km",
    "distance_edges_km",
    "heldout_group",
}

if MISSING_PKGS or not (PROJECT_ROOT / "Origindata").exists():
    print("⏭️ 跳过联合 HDF5 构建（当前为干跑预览模式）。")
    for mode in MOE_LEARN_MODES:
        cfg_preview = tomllib.loads(MOE_LEARN_TOML_MAP[mode].read_text(encoding="utf-8"))
        ds_preview = {k: v for k, v in cfg_preview["dataset"].items() if k not in MOE_LEARN_STRIP_KEYS}
        print(f"   [预览 MoE_learn:{mode}] split={ds_preview['split_strategy']}, window={ds_preview['segment_start_s']}..{ds_preview['segment_end_s']}s")
else:
    route_dirs = {k: Path(v) for k, v in GENERATED_LOCAL.get("base_routes", {}).get("routes", {}).items()}

    # 1) joint_feature_pack (7 种组合 x 指定融合策略)
    if JOINT_FUSIONS and route_dirs:
        from scripts.data_processing.joint_feature_pack.joint_features import build_joint_datasets

        GENERATED_LOCAL["joint_feature_pack"] = {}
        for fusion in JOINT_FUSIONS:
            target_root = LOCAL_OUTPUTS / "JointDatasets" / f"{dataset_tag}_{fusion}"
            if OVERWRITE_LOCAL_JOINTS and target_root.exists():
                shutil.rmtree(target_root)
            if not target_root.exists():
                built = build_joint_datasets(route_dirs, target_root, fusion_strategy=fusion)
            else:
                built = {p.name.split("_")[0]: p for p in sorted(target_root.iterdir()) if p.is_dir()}
            GENERATED_LOCAL["joint_feature_pack"][fusion] = {k: str(v) for k, v in built.items()}
            print(f"✅ joint_feature_pack [{fusion}] 7 组特征已就绪 -> {target_root.relative_to(PROJECT_ROOT)}")

    # 2) MoE_data_scale (RBD_SCM)
    if BUILD_MOE_RBD_SCM and route_dirs:
        from scripts.data_processing.MoE_data_scale.joint_features import build_joint_dataset as build_moe_joint

        moe_target = LOCAL_OUTPUTS / "MoE_JointDatasets" / dataset_tag
        if OVERWRITE_LOCAL_JOINTS and moe_target.exists():
            shutil.rmtree(moe_target)
        built_moe = build_moe_joint(route_dirs, moe_target)
        GENERATED_LOCAL["MoE_data_scale"] = {k: str(v) for k, v in built_moe.items()}
        print(f"✅ MoE_data_scale [RBD_SCM] 已就绪 -> {Path(built_moe['RBD_SCM']).relative_to(PROJECT_ROOT)}")

    # 3) MoE_learn (逐窗去均值 RBD_SCM_train.h5 / RBD_SCM_test.h5)
    if MOE_LEARN_MODES:
        from scripts.data_processing.MoE_learn.generation_config import normalize_config as norm_moe_learn
        from scripts.data_processing.MoE_learn.generation_core import generate_datasets as gen_moe_learn
        from scripts.data_processing.MoE_learn.joint_features import build_joint_dataset as build_moe_learn_joint

        GENERATED_LOCAL["MoE_learn"] = {}
        for mode in MOE_LEARN_MODES:
            raw_ml = tomllib.loads(MOE_LEARN_TOML_MAP[mode].read_text(encoding="utf-8"))
            ml_dataset = {
                k: deepcopy(v)
                for k, v in raw_ml["dataset"].items()
                if k not in MOE_LEARN_STRIP_KEYS
            }
            if QUICK_SMOKE_END_S is not None and mode != "demo":
                ml_dataset["segment_end_s"] = int(QUICK_SMOKE_END_S)
                ml_dataset["manual_dataset_variant_tag"] = f"moe_learn_smoke_{int(QUICK_SMOKE_END_S)}s"
            ml_norm = norm_moe_learn(
                {
                    "schema_version": 1,
                    "dataset": ml_dataset,
                    "rbd": deepcopy(raw_ml["rbd"]),
                    "result_path": f"outputs/MoE_learn/{mode}/generation-result.json",
                },
                root=PROJECT_ROOT,
            )
            ml_routes = gen_moe_learn(ml_norm, root=PROJECT_ROOT)
            ml_joint_root = LOCAL_OUTPUTS / "MoE_learn" / mode / "joint"
            if OVERWRITE_LOCAL_JOINTS and ml_joint_root.exists():
                shutil.rmtree(ml_joint_root)
            ml_joint = build_moe_learn_joint(ml_routes["datasets"], ml_joint_root)
            rbd_scm_dir = Path(ml_joint["RBD_SCM"])
            GENERATED_LOCAL["MoE_learn"][mode] = {
                "dir": str(rbd_scm_dir),
                "train_h5": str(rbd_scm_dir / "RBD_SCM_train.h5"),
                "test_h5": str(rbd_scm_dir / "RBD_SCM_test.h5"),
            }
            print(f"✅ MoE_learn [{mode}] 已就绪 -> {rbd_scm_dir.relative_to(PROJECT_ROOT)}")

⏭️ 跳过联合 HDF5 构建（当前为干跑预览模式）。
   [预览 MoE_learn:demo] split=full_interleaved, window=0..119s
   [预览 MoE_learn:before_then_after] split=range_overlap, window=0..4499s

## 4. 同步 HDF5 到 Google Drive 并写入 `dataset_manifest.json`

将本地 SSD 生成完毕的数据集整目录复制到 `DRIVE_DATASET_ROOT`，并写入相对路径清单 `dataset_manifest.json`。完成后即可**关闭并释放当前 CPU 运行时**，切换到 **模块二（GPU）**。

In [5]:
# Cell 5: 将生成好的 HDF5 数据集复制到 Google Drive 并更新 dataset_manifest.json
from datetime import datetime, timezone
import hashlib
import json

MANIFEST_PATH = DRIVE_DATASET_ROOT / "dataset_manifest.json"


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def sync_dir_to_drive(src_dir: Path, rel_dest: Path) -> dict:
    dst_dir = DRIVE_DATASET_ROOT / rel_dest
    dst_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
    h5_files = {}
    for h5_file in sorted(dst_dir.glob("*.h5")):
        h5_files[h5_file.name] = {
            "size_mb": round(h5_file.stat().st_size / (1024 * 1024), 3),
            "sha256": sha256_file(h5_file),
        }
    return {
        "rel_dir": rel_dest.as_posix(),
        "files": h5_files,
    }


if not GENERATED_LOCAL:
    print("ℹ️ 本次未产生新的本地 HDF5 目录，展示 Google Drive 目标结构示例:")
    example_layout = {
        "drive_dataset_root": str(DRIVE_DATASET_ROOT),
        "manifest_file": str(MANIFEST_PATH),
        "entries": {
            f"presets/{dataset_tag}/base_routes/<RBD|ELM|SCM>": "*_train.h5, *_test.h5",
            f"presets/{dataset_tag}/joint_block_rms/<7 methods>": "*_train.h5, *_test.h5",
            f"presets/{dataset_tag}/moe_rbd_scm/RBD_SCM": "RBD_SCM_train.h5, RBD_SCM_test.h5",
            "moe_learn/<demo|before_then_after>/RBD_SCM": "RBD_SCM_train.h5, RBD_SCM_test.h5",
        },
    }
    print(json.dumps(example_layout, ensure_ascii=False, indent=2))
else:
    DRIVE_DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    manifest = (
        json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
        if MANIFEST_PATH.exists()
        else {"schema_version": 1, "presets": {}, "moe_learn": {}}
    )
    preset_entry = manifest["presets"].setdefault(dataset_tag, {})
    preset_entry["updated_at_utc"] = datetime.now(timezone.utc).isoformat()

    if "base_routes" in GENERATED_LOCAL:
        preset_entry["base_routes"] = {}
        for method, src_str in GENERATED_LOCAL["base_routes"]["routes"].items():
            src = Path(src_str)
            preset_entry["base_routes"][method] = sync_dir_to_drive(
                src, Path("presets") / dataset_tag / "base_routes" / method
            )

    if "joint_feature_pack" in GENERATED_LOCAL:
        preset_entry.setdefault("joint_feature_pack", {})
        for fusion, method_map in GENERATED_LOCAL["joint_feature_pack"].items():
            preset_entry["joint_feature_pack"][fusion] = {}
            for method, src_str in method_map.items():
                src = Path(src_str)
                preset_entry["joint_feature_pack"][fusion][method] = sync_dir_to_drive(
                    src, Path("presets") / dataset_tag / f"joint_{fusion}" / method
                )

    if "MoE_data_scale" in GENERATED_LOCAL:
        preset_entry["MoE_data_scale"] = {}
        for method, src_str in GENERATED_LOCAL["MoE_data_scale"].items():
            src = Path(src_str)
            preset_entry["MoE_data_scale"][method] = sync_dir_to_drive(
                src, Path("presets") / dataset_tag / "moe_rbd_scm" / method
            )

    if "MoE_learn" in GENERATED_LOCAL:
        for mode, info in GENERATED_LOCAL["MoE_learn"].items():
            src = Path(info["dir"])
            manifest["moe_learn"][mode] = sync_dir_to_drive(
                src, Path("moe_learn") / mode / "RBD_SCM"
            )

    MANIFEST_PATH.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    print(f"🎉 已全部同步至 Google Drive 并更新清单: {MANIFEST_PATH}")
    print("👉 现在可以断开当前 CPU 运行时，打开 notebooks/02_network_training_gpu.ipynb 连接 GPU 开始训练！")

ℹ️ 本次未产生新的本地 HDF5 目录，展示 Google Drive 目标结构示例:
{
  "drive_dataset_root": "/Volumes/exDateDisk/projects/SwellEx/outputs/drive_staging/datasets",
  "manifest_file": "/Volumes/exDateDisk/projects/SwellEx/outputs/drive_staging/datasets/dataset_manifest.json",
  "entries": {
    "presets/train_before/base_routes/<RBD|ELM|SCM>": "*_train.h5, *_test.h5",
    "presets/train_before/joint_block_rms/<7 methods>": "*_train.h5, *_test.h5",
    "presets/train_before/moe_rbd_scm/RBD_SCM": "RBD_SCM_train.h5, RBD_SCM_test.h5",
    "moe_learn/<demo|before_then_after>/RBD_SCM": "RBD_SCM_train.h5, RBD_SCM_test.h5"
  }
}